# AutoEncoder

## 3.1 Part 1: Autoencoder for Image Reconstruction

### 3.1.1 Autoencoder Implementation

In [ ]:
import numpy as np
import os
import requests
import matplotlib.pyplot as plt
import random
import hashlib
import tqdm
from tqdm import tqdm 
import math
import time
from collections import Counter
import pandas as pd
import gzip
from PIL import Image, ImageDraw, ImageFont
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, roc_curve, auc, precision_recall_curve
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_curve, auc, precision_recall_curve, f1_score, precision_score, recall_score

In [ ]:
username = "gracy.garg"
# seed = 42
seed = int ( hashlib . sha256 ( username . encode () ) . hexdigest () , 16)
seed = seed % (2**32)
np.random.seed(seed)
random.seed(seed)
# print(f"Seed for {username}: {seed}")

In [ ]:
def add_username_to_plot(username="gracy.garg"):
    """
    Adds the username to the top-right corner of the current plot.
    """
    plt.text(
        0.95, 0.95, username,
        ha='right', va='top',
        transform=plt.gca().transAxes,
        fontsize=10, color='gray', alpha=0.7
    )

In [ ]:
import os
import matplotlib.pyplot as plt

def save_plot_to_output_folder(file_name, output_dir="q3_output"):
    """
    Saves the currently active Matplotlib plot to a specified folder 
    and appends the .png extension if not present.

    Args:
        file_name (str): The custom name for the file 
        output_dir (str): The target directory to save the file (default: "q3_output").
    """
    # 1. Ensure the directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # 2. Add .png extension if missing
    if not file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        file_name += ".png"
        
    # 3. Construct the full file path
    full_path = os.path.join(output_dir, file_name)
    
    # 4. Save the currently active figure
    try:
        plt.savefig(full_path, bbox_inches='tight')
        print(f"\nPlot successfully saved to: {full_path}")
    except Exception as e:
        print(f"\nERROR saving plot to {full_path}: {e}")

### 3.1.2 Training and Visualization on MNIST 

In [ ]:
class Layer:
    def forward(self, x): raise NotImplementedError
    def backward(self, grad): raise NotImplementedError
    def zero_grad(self): pass
    def update(self, lr): pass

class Activation(Layer): pass

class ReLU(Activation):
    def forward(self, z: np.ndarray) -> np.ndarray: self.z = z; return np.maximum(0, z)
    def backward(self, grad_out: np.ndarray) -> np.ndarray: return grad_out * (self.z > 0)

class Sigmoid(Activation):
    def forward(self, z: np.ndarray) -> np.ndarray:
        # --- FIX 1: Add clipping for numerical stability ---
        # Prevents np.exp() from overflowing with large negative inputs.
        z = np.clip(z, -500, 500)
        self.out = 1 / (1 + np.exp(-z))
        return self.out
    def backward(self, grad_out: np.ndarray) -> np.ndarray: return grad_out * self.out * (1 - self.out)

class Identity(Activation):
    def forward(self, z: np.ndarray) -> np.ndarray: return z
    def backward(self, grad_out: np.ndarray) -> np.ndarray: return grad_out

class Linear(Layer):
    def __init__(self, in_dim, out_dim, activation=None):
        self.W = np.random.randn(in_dim, out_dim) * np.sqrt(2.0 / in_dim)
        self.b = np.zeros(out_dim)
        self.activation = activation; self.dW = np.zeros_like(self.W); self.db = np.zeros_like(self.b)
    def forward(self, x): self.x = x; self.z = x @ self.W + self.b; return self.activation.forward(self.z) if self.activation else self.z
    def backward(self, grad):
        if self.activation: grad = self.activation.backward(grad)
        # We calculate gradients for this batch, not accumulate
        self.dW = self.x.T @ grad
        self.db = np.sum(grad, axis=0)
        return grad @ self.W.T
    def update(self, lr): self.W -= lr * self.dW; self.b -= lr * self.db

class Model:
    def __init__(self, layers): self.layers = layers
    def forward(self, X):
        for layer in self.layers: X = layer.forward(X)
        return X
    def backward(self, grad):
        for layer in reversed(self.layers): grad = layer.backward(grad)
        return grad
    def update_params(self, lr):
        for layer in self.layers:
            if isinstance(layer, Linear): layer.update(lr)

In [ ]:

# --- Loss Functions ---
def mse_loss(y_pred, y_true): 
    loss = np.mean((y_pred - y_true)**2)
    grad = 2 * (y_pred - y_true) / y_true.shape[0]
    return loss, grad

def mse(y_pred, y_true):
    return np.mean((y_pred - y_true)**2)

def mse_grad(y_pred, y_true):
    return 2 * (y_pred - y_true) / y_true.shape[0]



In [ ]:
class MLPAutoencoder:
    def __init__(self, input_dim: int, latent_dim: int, hidden_layers: list = None):
        if hidden_layers is None: hidden_layers = [128]
        encoder_arch = [input_dim] + hidden_layers + [latent_dim]
        encoder_layers = [Linear(encoder_arch[i], encoder_arch[i+1], activation=ReLU() if i < len(encoder_arch)-2 else Identity()) for i in range(len(encoder_arch)-1)]
        self.encoder = Model(encoder_layers); print(f"Encoder Architecture: {encoder_arch}")
        decoder_arch = [latent_dim] + list(reversed(hidden_layers)) + [input_dim]
        decoder_layers = [Linear(decoder_arch[i], decoder_arch[i+1], activation=ReLU() if i < len(decoder_arch)-2 else Sigmoid()) for i in range(len(decoder_arch)-1)]
        self.decoder = Model(decoder_layers); print(f"Decoder Architecture: {decoder_arch}")
    def forward(self, x: np.ndarray) -> np.ndarray: return self.decoder.forward(self.encoder.forward(x))

In [ ]:
def fetch_and_load_mnist(path="data"):
    base_url = "https://ossci-datasets.s3.amazonaws.com/mnist/"
    files = {
        'train_images': 'train-images-idx3-ubyte.gz',
        'train_labels': 'train-labels-idx1-ubyte.gz',
        'test_images': 't10k-images-idx3-ubyte.gz',
        'test_labels': 't10k-labels-idx1-ubyte.gz'
    }
    os.makedirs(path, exist_ok=True)
    for filename in files.values():
        filepath = os.path.join(path, filename)
        if not os.path.exists(filepath):
            print(f"Downloading {filename}...")
            response = requests.get(base_url + filename); response.raise_for_status()
            with open(filepath, 'wb') as f: f.write(response.content)

    def load_images(filename):
        with gzip.open(os.path.join(path, filename), 'rb') as f:
            data = np.frombuffer(f.read(), dtype=np.uint8, offset=16)
        return data.reshape(-1, 784)
    
    def load_labels(filename):
        with gzip.open(os.path.join(path, filename), 'rb') as f:
            return np.frombuffer(f.read(), dtype=np.uint8, offset=8)
    X_train = load_images(files['train_images']); y_train = load_labels(files['train_labels'])
    X_test = load_images(files['test_images']); y_test = load_labels(files['test_labels'])
    return X_train, y_train, X_test, y_test

### 3.1.2 Training and Visualization on MNIST

In [ ]:

# -- Part 4: Main Execution Block ---
INPUT_DIM = 784
LATENT_DIM = 32
HIDDEN_LAYERS = [1000,256, 128]
LEARNING_RATE = 0.01
EPOCHS = 20
BATCH_SIZE = 64

In [ ]:
# print("\n--- Loading MNIST Data ---")
# x_train, y_train, x_test, y_test = fetch_and_load_mnist()
# x_train = x_train.astype('float32') / 255.; x_test = x_test.astype('float32') / 255.
# autoencoder = MLPAutoencoder(input_dim=INPUT_DIM, latent_dim=LATENT_DIM, hidden_layers=HIDDEN_LAYERS)
# print("\n--- Starting Training ---")
# for epoch in range(EPOCHS):
#     epoch_loss = 0.0; permutation = np.random.permutation(x_train.shape[0])
#     for i in range(0, x_train.shape[0], BATCH_SIZE):
#         indices = permutation[i:i + BATCH_SIZE]; batch_x = x_train[indices]
#         reconstruction = autoencoder.forward(batch_x)
#         loss, grad = mse_loss(reconstruction, batch_x)
#         grad = autoencoder.decoder.backward(grad); autoencoder.encoder.backward(grad)
#         autoencoder.encoder.update_params(LEARNING_RATE); autoencoder.decoder.update_params(LEARNING_RATE)
#         epoch_loss += loss * len(batch_x)
#     print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {epoch_loss / x_train.shape[0]:.6f}")
    
# print("--- Training Finished ---\n")

In [ ]:
# print("--- Visualizing Results ---")
# unique_digits = {}; [unique_digits.setdefault(y_test[i], x_test[i]) for i in range(len(x_test)) if len(unique_digits) < 10]
# original_images = np.array([unique_digits[i] for i in range(10)])
# decoded_imgs = autoencoder.forward(original_images)
# n = 10; plt.figure(figsize=(20, 6))
# add_username_to_plot(username)
# for i in range(n):
#     ax = plt.subplot(2, n, i + 1); plt.imshow(original_images[i].reshape(28, 28), cmap='gray'); plt.title(f"Original: {i}"); plt.axis('off')
#     ax = plt.subplot(2, n, i + 1 + n); plt.imshow(decoded_imgs[i].reshape(28, 28), cmap='gray'); plt.title(f"Recon: {i}"); plt.axis('off')
# plt.suptitle("Reconstruction of All Digits (Corrected)", fontsize=16)
# save_plot_to_output_folder("mnist_digits")
# plt.show()

## 3.2 Anomaly Detection with Autoencoders


In [ ]:
def load_and_inspect_image_sizes(folder_path):
    """
    Loads all images from the LFW folder without any resizing or filtering.
    It then counts and reports all the unique dimensions found.
    """
    if not os.path.isdir(folder_path):
        raise ValueError(f"The provided path '{folder_path}' is not a valid directory.")

    print(f"Inspecting all images in local path: {folder_path}")

    image_dimensions = []
    person_folders = sorted([d for d in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, d))])

    total_images_scanned = 0
    for person_name in person_folders:
        person_path = os.path.join(folder_path, person_name)
        image_files = [f for f in os.listdir(person_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for image_file in image_files:
            image_path = os.path.join(person_path, image_file)


            with Image.open(image_path) as img:
                image_dimensions.append(img.size)

            total_images_scanned += 1

    print(f"\nScan complete. Inspected a total of {total_images_scanned} images.")

    size_counts = Counter(image_dimensions)

    print("\n--- Found the following image dimensions ---")
    print("Format: (Width, Height): Number of images")

    for size, count in sorted(size_counts.items(), key=lambda item: item[1], reverse=True):
        print(f"{size}: {count} images")

    return size_counts

In [ ]:
LFW_FOLDER_PATH = "LFW_Dataset"
try:
    load_and_inspect_image_sizes(folder_path=LFW_FOLDER_PATH)
except ValueError as e:
    print(f"\nError: {e}")
    print("Please update the 'LFW_FOLDER_PATH' variable before running.")


In [ ]:
# Define Optimizer
class Adam:         
    def __init__(self, model, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.model = model
        self.lr = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.t = 0
        self.layers = self.model.encoder.layers + self.model.decoder.layers
        self.m = [[np.zeros_like(l.W), np.zeros_like(l.b)] for l in self.layers]
        self.v = [[np.zeros_like(l.W), np.zeros_like(l.b)] for l in self.layers]

    def step(self):
        self.t += 1
        for i, layer in enumerate(self.layers):
            grad_W, grad_b = layer.dW, layer.db
            self.m[i][0] = self.beta1 * self.m[i][0] + (1 - self.beta1) * grad_W
            self.v[i][0] = self.beta2 * self.v[i][0] + (1 - self.beta2) * (grad_W ** 2)
            m_hat = self.m[i][0] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i][0] / (1 - self.beta2 ** self.t)
            layer.W -= self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)
            self.m[i][1] = self.beta1 * self.m[i][1] + (1 - self.beta1) * grad_b
            self.v[i][1] = self.beta2 * self.v[i][1] + (1 - self.beta2) * (grad_b ** 2)
            m_hat_b = self.m[i][1] / (1 - self.beta1 ** self.t)
            v_hat_b = self.v[i][1] / (1 - self.beta2 ** self.t)
            layer.b -= self.lr * m_hat_b / (np.sqrt(v_hat_b) + self.epsilon)


In [ ]:
# Define Autoencoder using the previously defined layers and model
class MLPAutoencoder:
    def __init__(self, input_dim: int, latent_dim: int, hidden_layers: list = None):
        if hidden_layers is None: hidden_layers = [128]
        encoder_arch = [input_dim] + hidden_layers + [latent_dim]
        decoder_arch = [latent_dim] + list(reversed(hidden_layers)) + [input_dim]
        encoder_layers = [Linear(encoder_arch[i], encoder_arch[i+1], activation=ReLU() if i < len(encoder_arch)-2 else Identity()) for i in range(len(encoder_arch)-1)]
        self.encoder = Model(encoder_layers)
        decoder_layers = [Linear(decoder_arch[i], decoder_arch[i+1], activation=ReLU() if i < len(decoder_arch)-2 else Sigmoid()) for i in range(len(decoder_arch)-1)]
        self.decoder = Model(decoder_layers)
        print(f"\nInitialized Autoencoder.\nEncoder: {encoder_arch}\nDecoder: {decoder_arch}")

    def forward(self, x: np.ndarray) -> np.ndarray:
        return self.decoder.forward(self.encoder.forward(x))


In [ ]:
def load_lfw_from_local_folder(folder_path):
    """
    Loads, resizes, and flattens images from a local directory.
    """
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"The directory '{folder_path}' was not found.")

    X_data, y_indices = [], []
    # Consistent image dimensions from the original script (resize=0.4)
    h, w = 250, 250

    # Get person names and create a mapping from name to an integer index
    target_names = [d for d in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, d))]
    target_names.sort()
    name_to_index = {name: i for i, name in enumerate(target_names)}

    print(f"Found {len(target_names)} people in '{folder_path}'. Loading images...")

    for person_name in target_names:
        person_folder = os.path.join(folder_path, person_name)
        person_index = name_to_index[person_name]

        for image_name in os.listdir(person_folder):
            if image_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_path = os.path.join(person_folder, image_name)
                # Open image, convert to grayscale, and resize
                img = Image.open(image_path).convert('L').resize((w, h))
                # Convert to numpy array and flatten
                X_data.append(np.array(img).flatten())
                y_indices.append(person_index)

    print(f"Loaded {len(X_data)} total images.")

    # Convert lists to numpy arrays
    X = np.array(X_data)
    y = np.array(y_indices)

    return X, y, np.array(target_names), h, w


In [ ]:
LFW_FOLDER_PATH = "LFW_Dataset"  

try:
    X, y, target_names, h, w = load_lfw_from_local_folder(LFW_FOLDER_PATH)
except FileNotFoundError as e:
    print(e)
    exit()

X = X.astype(np.float32) / 255.0
bush_class_index = list(target_names).index('George_W_Bush')
X_train_normal = X[y == bush_class_index]
print(f"Step 1  Filtered data to {len(X_train_normal)} 'normal' (George W Bush) images.")

input_dimension = h * w
autoencoder = MLPAutoencoder(input_dim=input_dimension, latent_dim=32, hidden_layers=[512, 128])
optimizer = Adam(autoencoder, learning_rate=0.001)

In [ ]:

epochs, batch_size = 25, 32
print("\nStep 2: Starting Model Training with Adam Optimizer...")

for epoch in range(epochs):
    permutation = np.random.permutation(X_train_normal.shape[0])
    epoch_loss = 0
    for i in range(0, X_train_normal.shape[0], batch_size):
        indices = permutation[i:i + batch_size]
        batch = X_train_normal[indices]
        reconstruction = autoencoder.forward(batch)
        loss = mse(reconstruction, batch)
        grad = mse_grad(reconstruction, batch)
        grad = autoencoder.decoder.backward(grad)
        autoencoder.encoder.backward(grad)
        optimizer.step()
        epoch_loss += loss * len(batch)

    print(f"  - Epoch {epoch+1}/{epochs}, Training Loss: {epoch_loss / len(X_train_normal):.6f}")
print("Training Complete.")

### Test 

In [ ]:
### Reconstructing a single 'normal' image to check performance.
test_image = X_train_normal[0:1]
reconstructed_image = autoencoder.forward(test_image)
error = mse(reconstructed_image, test_image)

fig, axes = plt.subplots(1, 2, figsize=(8, 5))
add_username_to_plot(username)
axes[0].imshow(test_image.reshape(h, w), cmap='gray'); axes[0].set_title("Original Image"); axes[0].axis('off')
axes[1].imshow(reconstructed_image.reshape(h, w), cmap='gray'); axes[1].set_title("Reconstructed Image"); axes[1].axis('off')
plt.suptitle(f"Single Image Reconstruction\nReconstruction Error (MSE): {error:.5f}", fontsize=14)
plt.show()

### 3.2.1 Anomaly Detection

In [ ]:
# The "test set" is the entire dataset.
X_test = X
# y_true_anomaly is our ground truth:
#  1 for an anomaly (not Bush),
#  0 for normal (Bush)
y_true_anomaly = (y != bush_class_index).astype(int)

# print("Calculating reconstruction errors for all images...")
# reconstructions = autoencoder.forward(X_test)
# # Calculate Mean Squared Error for each image individually
# errors = np.mean((reconstructions - X_test)**2, axis=1)

In [ ]:
# # --- 1. Calculate AUC Score ---
# fpr, tpr, roc_thresholds = roc_curve(y_true_anomaly, errors)
# auc_score = auc(fpr, tpr)
# print(f"AUC Score calculated.")


In [ ]:
# # --- 2. Determine a Suitable Threshold ---
# precisions, recalls, pr_thresholds = precision_recall_curve(y_true_anomaly, errors)

# # Calculate F1 score for each threshold, handling the case where precision+recall is 0
# f1_scores = (2 * precisions * recalls) / (precisions + recalls)
# f1_scores = np.nan_to_num(f1_scores) # Handle division by zero

# # Find the threshold that corresponds to the highest F1 score
# best_f1_idx = np.argmax(f1_scores)
# best_threshold = pr_thresholds[best_f1_idx]
# print(f"Suitable threshold determined.")

In [ ]:
# # --- 3. Calculate Metrics at the Optimal Threshold ---
# # Classify as anomaly (1) if the error is greater than our threshold
# y_pred_anomaly = (errors > best_threshold).astype(int)

# # Calculate precision, recall, and F1-score
# precision = precision_score(y_true_anomaly, y_pred_anomaly)
# recall = recall_score(y_true_anomaly, y_pred_anomaly)
# f1 = f1_score(y_true_anomaly, y_pred_anomaly)
# print(f"Precision, Recall, and F1-Score calculated.")


In [ ]:
# # --- 4. Report the Results ---
# print("\n--- Anomaly Detection Performance ---")
# print(f"AUC Score: {auc_score:.4f}")
# print("---------------------------------------")
# print(f"Best Threshold (Maximizing F1): {best_threshold:.6f}")
# print(f"Precision: {precision:.4f}")
# print(f"Recall:    {recall:.4f}")
# print(f"F1-Score:  {f1:.4f}")
# print("---------------------------------------")

# plt.figure(figsize=(8, 6))
# plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {auc_score:.4f})')
# plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance')
# plt.xlim([0.0, 1.0])
# plt.ylim([0.0, 1.05])
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('Receiver Operating Characteristic (ROC) Curve')
# plt.legend(loc="lower right")
# plt.grid(True)
# add_username_to_plot(username)
# save_plot_to_output_folder("lfw_roc_curve")
# plt.show()


In [ ]:
# Identify indices for 'Bush' and 'Non-Bush' images
bush_indices = np.where(y == bush_class_index)[0]
non_bush_indices = np.where(y != bush_class_index)[0]

# Select 1 random Bush image
np.random.seed(42) # for reproducibility
bush_sample_idx = np.random.choice(bush_indices, 1)[0]

# Select 5 random Non-Bush images
non_bush_sample_indices = np.random.choice(non_bush_indices, 5, replace=False)

# Combine indices
all_sample_indices = np.concatenate(([bush_sample_idx], non_bush_sample_indices))

# Get titles
plot_titles = [f"George W. Bush (Normal)"] + [f"{target_names[y[idx]]} (Anomaly)" for idx in non_bush_sample_indices]

# Calculate reconstructions and errors
selected_X = X[all_sample_indices]
selected_reconstructions = autoencoder.forward(selected_X)
selected_errors = np.mean((selected_reconstructions - selected_X)**2, axis=1)


print("\n--- Reconstruction Errors for Visualized Samples ---")
for i in range(len(all_sample_indices)):
    status = "Normal" if i == 0 else "Anomaly"
    print(f"- {plot_titles[i]:<35} (Status: {status}) | Error = {selected_errors[i]:.4f}")


fig, axes = plt.subplots(2, 6, figsize=(20, 7))
fig.suptitle('', fontsize=16)
# add_username_to_plot(username)
for i, idx in enumerate(all_sample_indices):
    # Top Row: Original Image
    axes[0, i].imshow(X[idx].reshape(h, w), cmap='gray')
    axes[0, i].set_title(f"{plot_titles[i]}\nOriginal", fontsize=10)
    axes[0, i].axis('off')

    # Bottom Row: Reconstructed Image with Error
    axes[1, i].imshow(selected_reconstructions[i].reshape(h, w), cmap='gray')
    axes[1, i].set_title(f"Reconstruction\nError: {selected_errors[i]:.4f}", fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95]) 
save_plot_to_output_folder("lfw_anomaly_reconstructions")
plt.show()

print("\nVisualization complete.")

In [ ]:
def plot_error_boxplot(bush_errors, other_errors, best_threshold):
    """
    Visualizes the distribution of reconstruction errors using a Box Plot.
    This clearly shows the median, quartiles, and range of Normal vs. Anomaly errors.
    """
    
    data_to_plot = [bush_errors, other_errors]
    labels = ['Normal (George W. Bush)', 'Anomaly (Others)']
    
    plt.figure(figsize=(9, 7))
    
    bp = plt.boxplot(
        data_to_plot, 
        vert=True,     
        patch_artist=True,
        labels=labels,
        showfliers=False # Hide outliers for a cleaner look (optional)
    )
    
    colors = ['red', 'blue']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
        
    # 3. Add a horizontal line to show the optimal threshold (tau)
    plt.axhline(best_threshold, color='green', linestyle='--', linewidth=2, 
                label=f'Optimal Threshold (tau={best_threshold:.4f})')
    
    # Add titles and labels
    plt.title('Distribution of Reconstruction Errors: Normal vs. Anomaly (Box Plot)', fontsize=14)
    plt.ylabel('Reconstruction Error (MSE)', fontsize=12)
    plt.xlabel('Identity Class', fontsize=12)
    plt.legend(loc='upper right')
    plt.grid(axis='y', alpha=0.5)
    
    add_username_to_plot(username)
    
    plt.tight_layout()
    save_plot_to_output_folder("lfw_reconstruction_error_boxplot")
    plt.show()


print("\nStep 6: Plotting reconstruction error distributions (Box Plot)...")
print("\nVisualization complete.")

In [ ]:

def train_autoencoder(X_train, latent_dim, h, w):
    """Helper function to train a single autoencoder model."""
    input_dim = h * w
    autoencoder = MLPAutoencoder(input_dim=input_dim, latent_dim=latent_dim, hidden_layers=[512, 128])
    optimizer = Adam(autoencoder, learning_rate=0.001)

    epochs, batch_size = 25, 32
    print(f"\n--- Training Autoencoder with Bottleneck Dim: {latent_dim} ---")
    for epoch in range(epochs):
        permutation = np.random.permutation(X_train.shape[0])
        for i in range(0, X_train.shape[0], batch_size):
            indices = permutation[i:i + batch_size]
            batch = X_train[indices]

            reconstruction = autoencoder.forward(batch)
            loss_grad = mse_grad(reconstruction, batch)
            grad = autoencoder.decoder.backward(loss_grad)
            autoencoder.encoder.backward(grad)
            optimizer.step()
    print("Training complete.")
    return autoencoder


### 3.2.2 Analysis and Visualization

### Bottleneck Dimension Analysis

In [ ]:
# --- Load and Prepare Data ---
LFW_FOLDER_PATH = "LFW_Dataset"
try:
    X_full, y_full, target_names, h, w = load_lfw_from_local_folder(LFW_FOLDER_PATH)
except FileNotFoundError as e:
    print(e); exit()

X_full = X_full.astype(np.float32) / 255.0
bush_class_index = list(target_names).index('George_W_Bush')

# Training data is ONLY the "normal" class
X_train_normal = X_full[y_full == bush_class_index]

# Ground truth for anomaly detection: 0 for normal (Bush), 1 for anomaly (not Bush)
y_true_anomaly = (y_full != bush_class_index).astype(int)


# --- Bottleneck Dimension Analysis ---
latent_dims = [128, 64, 32]

# MODIFIED: Prepare a dictionary to ONLY store the best results
best_result = {'auc': -1, 'dim': None, 'model': None, 'errors': None}

plt.figure(figsize=(10, 8))
add_username_to_plot(username=username)

for ld in latent_dims:
    # Train the model
    model = train_autoencoder(X_train_normal, latent_dim=ld, h=h, w=w)

    # MODIFIED: Evaluate in batches to save RAM
    print(f"Evaluating model with dim={ld} in batches...")
    batch_size_eval = 256 # A larger batch size for faster evaluation
    all_errors = []
    for i in range(0, X_full.shape[0], batch_size_eval):
        batch_x = X_full[i:i + batch_size_eval]
        batch_recons = model.forward(batch_x)
        batch_errors = np.mean((batch_recons - batch_x)**2, axis=1)
        all_errors.extend(batch_errors)

    errors = np.array(all_errors)

    # Calculate ROC curve and AUC
    fpr, tpr, thresholds = roc_curve(y_true_anomaly, errors)
    auc_score = auc(fpr, tpr)

    if auc_score > best_result['auc']:
        best_result['auc'] = auc_score
        best_result['dim'] = ld
        best_result['model'] = model
        best_result['errors'] = errors
        print(f"New best model found! Dim: {ld}, AUC: {auc_score:.4f}")

    plt.plot(fpr, tpr, label=f'Bottleneck Dim = {ld} (AUC = {auc_score:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Different Bottleneck Dimension Analysis')
plt.legend()
plt.grid(True)
save_plot_to_output_folder("roc")
plt.show()


In [ ]:
print("--- Plotting 10 Random 1x3 Examples (with at least one George W. Bush) ---")

# --- 1. Select 1 Bush photo + 9 other random photos ---
num_examples = 10
total_samples = X_full.shape[0]

bush_indices = np.where(y_full == bush_class_index)[0]

bush_sample_idx = np.random.choice(bush_indices, 1)

all_indices = np.arange(total_samples)
other_indices_pool = np.setdiff1d(all_indices, bush_sample_idx)

other_random_indices = np.random.choice(other_indices_pool, num_examples - 1, replace=False)

# Combine and shuffle the final list of indices
final_indices = np.concatenate((bush_sample_idx, other_random_indices))
np.random.shuffle(final_indices)

In [ ]:
# --- 2. Loop through and plot each of the 10 selected examples ---
for idx in final_indices:
    # Get the data for the current sample
    original_image = X_full[idx].reshape(h, w)
    reconstructed_image = recons_full[idx].reshape(h, w)
    total_error = errors[idx]
    person_name = target_names[y_full[idx]]

    # Determine if it's the 'Normal' or 'Anomaly' class
    status = "Normal" if y_full[idx] == bush_class_index else "Anomaly"

    # Calculate the pixel-wise absolute error map
    error_map = np.abs(original_image - reconstructed_image)

    # Create the 1x3 plot for this person
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    add_username_to_plot(username)
    fig.suptitle(f'Example: {person_name} ({status}) | Total Error: {total_error:.4f}', fontsize=16)

    # Plot Original Image
    axes[0].imshow(original_image, cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    # Plot Reconstructed Image
    axes[1].imshow(reconstructed_image, cmap='gray')
    axes[1].set_title('Reconstruction')
    axes[1].axis('off')

    # Plot Error Map
    error_plot = axes[2].imshow(error_map, cmap='hot')
    axes[2].set_title('Pixel-wise Abs. Error')
    axes[2].axis('off')

    fig.colorbar(error_plot, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    save_plot_to_output_folder(f"lfw_example_{person_name.replace(' ', '_')}")
    plt.show()

print("\nPlotting complete.")

In [ ]:
best_dim = best_result['dim']
best_model = best_result['model']
errors = best_result['errors']
print(f"\nBest performing model has bottleneck dimension: {best_dim} with AUC = {best_result['auc']:.4f}")

# --- Determine a suitable threshold ---
# Find the threshold that maximizes the F1-score
precisions, recalls, thresholds_pr = precision_recall_curve(y_true_anomaly, errors)
# The last value of precisions and recalls can be problematic, so we slice them
f1_scores = (2 * precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1])
f1_scores = np.nan_to_num(f1_scores) # Handle division by zero
best_threshold_idx = np.argmax(f1_scores)
best_threshold = thresholds_pr[best_threshold_idx]

In [ ]:
# --- Calculate and Report Metrics ---
y_pred_anomaly = (errors > best_threshold).astype(int)

precision = precision_score(y_true_anomaly, y_pred_anomaly)
recall = recall_score(y_true_anomaly, y_pred_anomaly)
f1 = f1_score(y_true_anomaly, y_pred_anomaly)

print("\n--- Anomaly Detection Metrics for Best Model ---")
print(f"Optimal Threshold (Maximizing F1): {best_threshold:.6f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("-------------------------------------------------")

In [ ]:
# --- Visualize Classifications ---
# Reconstruct all images just once for visualization
recons_full = best_model.forward(X_full)

# The logic uses y_true_anomaly (1=Anomaly, 0=Normal)

tn_mask = (y_true_anomaly == 0) & (y_pred_anomaly == 0) 
tp_mask = (y_pred_anomaly == 1) & (y_true_anomaly == 1) 
fp_mask = (y_pred_anomaly == 1) & (y_true_anomaly == 0)
fn_mask = (y_pred_anomaly == 0) & (y_true_anomaly == 1)

# Ensure your index assignment uses these masks correctly.

# Get one example for each, ensuring we don't get an index error if a category is empty
idx_tn = np.where(tn_mask)[0][0] if np.any(tn_mask) else None
idx_tp = np.where(tp_mask)[0][0] if np.any(tp_mask) else None
idx_fp = np.where(fp_mask)[0][0] if np.any(fp_mask) else None
idx_fn = np.where(fn_mask)[0][0] if np.any(fn_mask) else None

examples = {
    'Correctly Normal (True Negative)': idx_tn,
    'Correctly Anomaly (True Positive)': idx_tp,
    'Misclassified Normal (False Positive)': idx_fp,
    'Misclassified Anomaly (False Negative)': idx_fn,
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
add_username_to_plot(username)
fig.suptitle('Comprehensive Visualization of Best Model Results', fontsize=20)

for i, (title, idx) in enumerate(examples.items()):
    if idx is None: # Handle cases where no example was found
        axes[0, i].set_title(f"{title}\n(No example found)")
        axes[0, i].axis('off')
        axes[1, i].axis('off')
        continue

    # Original Image
    axes[0, i].imshow(X_full[idx].reshape(h, w), cmap='gray')
    axes[0, i].set_title(f"{title}\nOriginal", fontsize=12)
    axes[0, i].axis('off')

    # Reconstructed Image
    axes[1, i].imshow(recons_full[idx].reshape(h, w), cmap='gray')
    axes[1, i].set_title(f"Reconstruction\nError: {errors[idx]:.4f}", fontsize=12)
    axes[1, i].axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.96])
save_plot_to_output_folder("lfw_best_model_classifications")
plt.show()

In [ ]:
# --- Plot PR Curve ---
plt.figure(figsize=(10, 8))
add_username_to_plot(username)
plt.plot(recalls, precisions, label=f'PR Curve (Best Model, Dim={best_dim})', color='purple')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for Best Model')
plt.grid(True)
plt.legend()
save_plot_to_output_folder("pr_curve")
plt.show()